In [1]:
%cd ..

/Users/sebastian/University/Master/third term/sem-proj/kg-token


/Users/sebastian/miniforge3/envs/kg-token/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import pandas as pd

In [3]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import torch
import numpy as np
from src.graph.Movielens100k import MovieLens
from torch_geometric.data import HeteroData

/Users/sebastian/miniforge3/envs/kg-token/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [11]:
uabase = pd.read_csv('data/ml-100k/' + 'ua.base', sep='\t', encoding='latin-1', header=None)
uabase.columns = ['user_id', 'movie_id', 'rating', 'timestamp']        

uatest = pd.read_csv('data/ml-100k/' + 'ua.test', sep='\t', header=None)
uatest.columns = ['user_id', 'movie_id', 'rating', 'timestamp']   


In [12]:
uabase

,user_id,movie_id,rating,timestamp
0,1,1,5,874965758
1,1,2,3,876893171
2,1,3,4,878542960
3,1,4,3,876893119
4,1,5,3,889751712
...,...,...,...,...
90565,943,1047,2,875502146
90566,943,1074,4,888640250
90567,943,1188,3,888640250
90568,943,1228,3,888640275


In [13]:
uatest

,user_id,movie_id,rating,timestamp
0,1,20,4,887431883
1,1,33,4,878542699
2,1,61,4,878542420
3,1,117,3,874965739
4,1,155,2,878542201
...,...,...,...,...
9425,943,232,4,888639867
9426,943,356,4,888639598
9427,943,570,1,888640125
9428,943,808,4,888639868


In [4]:
movielens = MovieLens(path='data/ml-100k/')
movielens.create_graph()

In [5]:
print(movielens.data["user", "likes", "movie"].edge_labels)

tensor([1., 1., 1.,  ..., 0., 0., 0.])


In [8]:
from src.models.graph.GraphModel import GraphTokenEncoder
from torch_geometric.nn import to_hetero
import torch_geometric.transforms as T
from torch_geometric.utils import structured_negative_sampling

In [9]:
EMBEDDING_DIM = 1024
model = GraphTokenEncoder(EMBEDDING_DIM, EMBEDDING_DIM)
model = to_hetero(model, movielens.data.metadata(), aggr='sum')

In [10]:
with torch.no_grad():  # Initialize lazy modules.
    out = model(movielens.data.x_dict, movielens.data.edge_index_dict)

In [11]:
out['movie'].shape

torch.Size([1682, 1024])

In [12]:
movielens.data.x_dict['user'].size(), movielens.data.x_dict['movie'].size()

(torch.Size([943, 1]), torch.Size([1682, 403]))

In [13]:
transform = T.RandomLinkSplit(
    num_val=0.1,  # 10% of edges for validation
    num_test=0.1,  # 10% of edges for test
    edge_types=('user', 'likes', 'movie'),
    rev_edge_types=('movie', 'rev_likes', 'user')
)
train_data, val_data, test_data = transform(movielens.data)

In [14]:
train_data.edge_index_dict['user', 'likes', 'movie']

tensor([[605, 307, 757,  ..., 767, 647, 708],
        [500, 708, 478,  ...,  15, 476, 317]])

In [15]:
structured_negative_sampling(train_data['user', 'likes', 'movie']['edge_index'], contains_neg_self_loops=False)

(tensor([605, 307, 757,  ..., 767, 647, 708]),
 tensor([500, 708, 478,  ...,  15, 476, 317]),
 tensor([ 374, 1480, 1459,  ..., 1230, 1668,  411]))

In [16]:
train_data['user', 'likes', 'movie']['edge_index']

tensor([[605, 307, 757,  ..., 767, 647, 708],
        [500, 708, 478,  ...,  15, 476, 317]])

In [17]:
train_data['user', 'likes', 'movie']['edge_index']

tensor([[605, 307, 757,  ..., 767, 647, 708],
        [500, 708, 478,  ...,  15, 476, 317]])

In [18]:
movielens.data

HeteroData(
  movie={ x=[1682, 403] },
  user={ x=[943, 1] },
  (user, likes, movie)={
    edge_index=[2, 100000],
    edge_labels=[100000],
  },
  (movie, rev_likes, user)={
    edge_index=[2, 100000],
    edge_labels=[100000],
  }
)

In [19]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [20]:
tokenizer("Hello this is me")

{'input_ids': [15496, 428, 318, 502], 'attention_mask': [1, 1, 1, 1]}

In [21]:
tokenizer.eos_token_id

50256

In [22]:
llm = AutoModelForCausalLM.from_pretrained("gpt2")

In [23]:
llm

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [29]:
text = 'X'
token_id = tokenizer.encode(text, add_special_tokens=False)  # 'X' -> [ID]

# Step 2: Convert token ID to a tensor
token_id_tensor = torch.tensor(token_id)  # Convert to tensor for PyTorch

# Step 3: Pass the token ID to the embedding layer to get the embedding
embedding = llm.transformer.wte(token_id_tensor)

In [30]:
embedding.shape

torch.Size([1, 768])

In [42]:
text = 'MOV'
token_id = tokenizer.encode(text, add_special_tokens=False)

In [43]:
token_id

[44, 8874]

In [45]:
special_tokens_dict = {'additional_special_tokens': ['<USER>','<MOVIE>']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
llm.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(50259, 768)

In [47]:
text = '<MOVIE>'
token_id = tokenizer.encode(text, add_special_tokens=False)
token_id

[50258]

In [52]:
llm.transformer.wte.weight[50258].shape

torch.Size([768])

In [56]:
tokens = tokenizer.encode("Q: Does user <USER> like movie <MOVIE>?\nA: ", return_tensors="pt")

In [59]:
tokens[0]

tensor([   48,    25,  8314,  2836,   220, 50257,   588,  3807,   220, 50258,
           30,   198,    32,    25,   220])

In [60]:
answer = tokenizer.encode("Yes", return_tensors="pt")[0]

In [61]:
answer

tensor([5297])

In [1]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
import os